# 04 · CTC sanity check (run before any training)
The notebook that saves days: id match across streams, strict length
alignment, the **T ≥ 2·L** feasibility check, one real forward/CTC/decode
pass, and an epoch-time estimate. **Gate:** zero hard mismatches, finite
loss, decode emits glosses.

In [ ]:
# --- Colab bootstrap (run first in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT = '/content/drive/MyDrive/cslr_phoenix/project'   # <- where this code lives
sys.path.append(PROJECT)
%cd $PROJECT

!pip -q install pyyaml
from src.utils import load_config
cfg = load_config('config.yaml')
print('config loaded:', cfg['project']['name'])


## Stream id coverage

In [ ]:
from src.utils import load_json, p
from pathlib import Path
for split in cfg['dataset']['splits']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    ids = {it['id'] for it in items}
    rgb = {f.stem for f in (p(cfg, 'features_rgb') / split).glob('*.npy')}
    kp  = {f.stem for f in (p(cfg, 'features_kp')  / split).glob('*.npy')}
    print(f'{split}: items {len(ids)} | rgb {len(rgb)} | kp {len(kp)} | '
          f'both {len(ids & rgb & kp)} | missing_rgb {len(ids-rgb)} | missing_kp {len(ids-kp)}')

## Length alignment + T ≥ 2·L feasibility (the CTC killer)

In [ ]:
import numpy as np
from src.utils import load_json, p
pool = cfg['model']['temporal_pool']
hard, infeasible = [], []
for split in cfg['dataset']['splits']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    for it in items:
        try:
            tr = np.load(p(cfg, 'features_rgb') / split / f"{it['id']}.npy", mmap_mode='r').shape[0]
            tk = np.load(p(cfg, 'features_kp')  / split / f"{it['id']}.npy", mmap_mode='r').shape[0]
        except FileNotFoundError:
            continue
        if abs(tr - tk) > 2:
            hard.append((split, it['id'], tr, tk))
        T = min(tr, tk) // pool
        if T < 2 * len(it['glosses']):
            infeasible.append((split, it['id'], T, len(it['glosses'])))
print('hard length mismatches (>2 frames):', len(hard), hard[:5])
print('T < 2L after pooling:', len(infeasible), infeasible[:5])
assert not hard, 'inspect hard mismatches before training (see src/data.MAX_LEN_MISMATCH)'
print('OK' if not infeasible else 'reduce temporal_pool or inspect the long-gloss samples above')

## One forward + CTC + greedy decode on the smoke split

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data import FeatureDataset, collate
from src.models import CSLRModel, count_params
from src.decode import greedy_decode, ids_to_glosses
from src.vocab import load_vocab
from src.utils import load_json, p, get_device

device = get_device()
g2i, i2g = load_vocab(p(cfg, 'manifests') / 'vocab.json')
items = load_json(p(cfg, 'manifests') / 'train.json')
smoke = load_json(p(cfg, 'manifests') / 'smoke_train_ids.json')

ds = FeatureDataset(cfg, items, 'train', ('rgb','kp'), g2i, train_mode=True, ids_subset=smoke)
dl = DataLoader(ds, batch_size=4, collate_fn=collate, shuffle=True)
batch = next(iter(dl))
print('batch ids:', batch['ids'])
print('padded rgb', tuple(batch['rgb'].shape), 'kp', tuple(batch['kp'].shape),
      'lengths', batch['lengths'].tolist())

model = CSLRModel(cfg, len(g2i)+1, ('rgb','kp')).to(device)
print('params:', count_params(model)/1e6, 'M')
out = model(batch, device)
crit = torch.nn.CTCLoss(blank=0, zero_infinity=cfg['train']['ctc_zero_infinity'])
loss = crit(out['main'], batch['targets'].to(device),
            out['out_lengths'].to(device), batch['target_lengths'].to(device))
print('CTC loss (finite?):', loss.item(), '->', 'OK' if torch.isfinite(loss) else 'BAD')

dec = ids_to_glosses(greedy_decode(out['main'], out['out_lengths']), i2g)
print('untrained greedy decode (gibberish expected):', dec[0][:10])

## Estimate epoch time (plan your Colab sessions)

In [ ]:
import time, torch
from torch.utils.data import DataLoader
full = FeatureDataset(cfg, items, 'train', ('rgb','kp'), g2i, train_mode=True)
loader = DataLoader(full, batch_size=cfg['train']['batch_size'], shuffle=True,
                    collate_fn=collate, num_workers=cfg['train']['num_workers'])
opt = torch.optim.Adam(model.parameters(), lr=cfg['train']['lr'])
model.train(); t0 = time.time(); N = 20
for i, b in enumerate(loader):
    if i >= N: break
    o = model(b, device)
    l = crit(o['main'], b['targets'].to(device), o['out_lengths'].to(device),
             b['target_lengths'].to(device))
    opt.zero_grad(); l.backward(); opt.step()
per_it = (time.time()-t0)/N
print(f'~{per_it:.2f}s/iter -> ~{per_it*len(loader)/60:.1f} min/epoch, '
      f'~{per_it*len(loader)*cfg["train"]["epochs"]/60:.0f} min for {cfg["train"]["epochs"]} epochs')